# FAN Algorithm ATPG — Structured Notebook

**Based on:** Fujiwara & Shimono, *"On the Acceleration of Test Generation Algorithms"*, IEEE Transactions on Computers, Vol. C-32, No. 12, December 1983.

This notebook implements the complete FAN ATPG algorithm with:
- Full 5-valued logic (0, 1, X, D, Dbar)
- Rules 1–7 from the paper
- SCOAP controllability-guided multiple backtrace
- Lecture-style (s, n0, n1) step-by-step trace output
- Brute-force oracle for result verification
- Seven benchmark circuits (lecture notes + paper figures)

---

## 1. Imports


In [1]:
import sys, re, itertools, copy
sys.setrecursionlimit(20000)

print("✅ Imports done.")


✅ Imports done.


## 2. 5-Valued Logic

FAN (like the D-Algorithm) operates in a **5-valued logic domain**:

| Symbol | Fault-free | Faulty | Meaning |
|--------|-----------|--------|---------|
| `0`    | 0 | 0 | Definite logic-0 everywhere |
| `1`    | 1 | 1 | Definite logic-1 everywhere |
| `X`    | ? | ? | Unassigned / don't care |
| `D`    | 1 | 0 | Fault effect: was 1, now 0 |
| `Dbar` | 0 | 1 | Inverted fault effect |

Gate functions are extended to operate on all five values.


In [2]:
ZERO = '0'
ONE  = '1'
X    = 'X'
D    = 'D'       # fault-free=1, faulty=0
DBAR = 'Dbar'    # fault-free=0, faulty=1

FAULT_FREE = {D: ONE,  DBAR: ZERO}
FAULTY     = {D: ZERO, DBAR: ONE}

def NOT5(a):
    return {ZERO: ONE, ONE: ZERO, D: DBAR, DBAR: D, X: X}[a]

def AND5(a, b):
    if ZERO in (a, b): return ZERO
    if a == X or b == X: return X
    if a == ONE: return b
    if b == ONE: return a
    if a == b:   return a
    return ZERO   # D & Dbar = 0

def OR5(a, b):
    if ONE in (a, b): return ONE
    if a == X or b == X: return X
    if a == ZERO: return b
    if b == ZERO: return a
    if a == b:    return a
    return ONE    # D | Dbar = 1

def AND_multi(*args):
    r = args[0]
    for v in args[1:]: r = AND5(r, v)
    return r

def OR_multi(*args):
    r = args[0]
    for v in args[1:]: r = OR5(r, v)
    return r

def NAND_multi(*args): return NOT5(AND_multi(*args))
def NOR_multi(*args):  return NOT5(OR_multi(*args))

def XOR5(a, b):
    if a == X or b == X: return X
    if a == ZERO: return b
    if b == ZERO: return a
    if a == ONE:  return NOT5(b)
    if b == ONE:  return NOT5(a)
    if a == b:    return ZERO
    return ONE

def XNOR5(*args):
    r = args[0]
    for v in args[1:]: r = XOR5(r, v)
    return NOT5(r)

def XOR_multi(*args):
    r = args[0]
    for v in args[1:]: r = XOR5(r, v)
    return r

LOGIC_FN = {
    'AND':  AND_multi,
    'OR':   OR_multi,
    'NAND': NAND_multi,
    'NOR':  NOR_multi,
    'XOR':  XOR_multi,
    'XNOR': XNOR5,
    'NOT':  lambda a: NOT5(a),
    'BUF':  lambda a: a,
}

print("✅ 5-valued logic functions defined.")


✅ 5-valued logic functions defined.


## 3. Circuit Model

Two classes represent the circuit:

- **`Gate`** — one node (primary input, logic gate, or primary output wire).  
- **`Circuit`** — the complete netlist container, with structural analysis
  (topological order, fan-out points, bound/free lines, headlines).

```
Gate  ──inputs──►  Gate  ──inputs──►  PO
```

`compute_topology()` performs (once, at parse time):
1. Topological sort  
2. Fan-out point detection  
3. Bound / free line classification  
4. Headline identification (Rule 4)


In [3]:
class Gate:
    def __init__(self, name, gtype, inputs):
        self.name   = name
        self.type   = gtype   # 'INPUT', 'AND', 'OR', ...
        self.inputs = inputs  # list of signal names
        self.value  = X

class Circuit:
    def __init__(self):
        self.gates         = {}   # name → Gate
        self.topo_order    = []   # topological order (computed once)
        self.fanout_points = set()# nodes that are fan-out points
        self.free_lines    = set()# lines not reachable from any fanout
        self.bound_lines   = set()# lines reachable from some fanout
        self.headlines     = set()# free lines adjacent to bound lines
        self.fault_node    = None
        self.stuck_val     = None

    # ── construction ─────────────────────────────────────────────
    def add_input(self, name):
        self.gates[name] = Gate(name, 'INPUT', [])

    def add_gate(self, name, gtype, inputs):
        self.gates[name] = Gate(name, gtype.upper(), inputs)

    # ── structural analysis ───────────────────────────────────────
    def compute_topology(self):
        """Topological sort + fan-out points + free/bound/headline sets."""
        # Topological sort
        visited, order = set(), []
        def visit(n):
            if n in visited: return
            g = self.gates[n]
            if g.type != 'INPUT':
                for i in g.inputs: visit(i)
            visited.add(n); order.append(n)
        for n in self.gates: visit(n)
        self.topo_order = order

        # Fan-out points: nodes used as input by more than one gate
        driver_count = {n: 0 for n in self.gates}
        for g in self.gates.values():
            for i in g.inputs:
                driver_count[i] += 1
        self.fanout_points = {n for n, c in driver_count.items() if c > 1}

        # Bound lines: reachable forward from any fan-out point
        self.bound_lines = set()
        for fp in self.fanout_points:
            self._mark_bound(fp)

        # Free lines
        self.free_lines = set(self.gates.keys()) - self.bound_lines

        # Headlines: free lines that have at least one bound input, or are PIs
        self.headlines = set()
        for n in self.free_lines:
            g = self.gates[n]
            if g.type == 'INPUT':
                continue  # PIs handled below
            for i in g.inputs:
                if i in self.bound_lines or i in self.fanout_points:
                    self.headlines.add(n)
                    break
        # PIs are the ultimate backtrace stop → treat as headlines
        for n in self.free_lines:
            if self.gates[n].type == 'INPUT':
                self.headlines.add(n)

    def _mark_bound(self, start):
        """Mark all nodes reachable forward from 'start' as bound."""
        driven_by = {n: [] for n in self.gates}
        for g in self.gates.values():
            for i in g.inputs:
                driven_by[i].append(g.name)
        stack = [start]
        while stack:
            cur = stack.pop()
            if cur in self.bound_lines: continue
            self.bound_lines.add(cur)
            stack.extend(driven_by.get(cur, []))

    def get_primary_outputs(self):
        driven = {i for g in self.gates.values() for i in g.inputs}
        return [n for n in self.gates if n not in driven]

    def get_primary_inputs(self):
        return [n for n, g in self.gates.items() if g.type == 'INPUT']

    # ── value access ──────────────────────────────────────────────
    def val(self, name):        return self.gates[name].value
    def set_val(self, name, v): self.gates[name].value = v
    def snapshot(self):         return {n: g.value for n, g in self.gates.items()}
    def restore(self, snap):
        for n, v in snap.items(): self.gates[n].value = v

    # ── evaluation ────────────────────────────────────────────────
    def evaluate_node(self, name):
        """Compute and update the value of a single non-input gate.
        CRITICAL: if this node is the fault node (has D or Dbar injected),
        NEVER overwrite it — that would erase the fault signal.
        """
        g = self.gates[name]
        if g.type == 'INPUT': return True
        fn = LOGIC_FN.get(g.type)
        if fn is None: return True
        # FAULT NODE PROTECTION
        if name == self.fault_node and g.value in (D, DBAR):
            return True   # injected by Rule 1 — never overwrite
        ins = [self.val(i) for i in g.inputs]
        computed = fn(*ins)
        existing = g.value
        if existing == X:
            g.value = computed
        elif computed != X and computed != existing:
            return False  # conflict
        return True

    def full_forward_implication(self):
        """Forward implication over the entire topological order."""
        for name in self.topo_order:
            g = self.gates[name]
            if g.type == 'INPUT': continue
            if not self.evaluate_node(name):
                return False
        return True

    def backward_implication(self):
        """
        RULE 2: Backward implication.
        If a gate output is known AND all-but-one input is known,
        deduce the unknown input. Repeat until stable.
        """
        changed = True
        while changed:
            changed = False
            for name in reversed(self.topo_order):
                g = self.gates[name]
                if g.type in ('INPUT', 'XOR', 'XNOR'): continue
                out_val = g.value
                if out_val == X: continue
                ins   = g.inputs
                ivals = [self.val(i) for i in ins]
                x_idxs = [j for j, v in enumerate(ivals) if v == X]
                if len(x_idxs) != 1: continue
                xi    = x_idxs[0]
                known = [ivals[j] for j in range(len(ins)) if j != xi]
                forced = self._backward_deduce(g.type, out_val, known)
                if forced is not None:
                    old = self.val(ins[xi])
                    if old == X:
                        self.set_val(ins[xi], forced); changed = True
                    elif old != forced:
                        return False  # conflict
        return True

    def _backward_deduce(self, gtype, out_val, known_others):
        """Deduce the one unknown input from gate type, output, and known inputs."""
        if gtype == 'AND':
            if out_val == ONE: return ONE
            if out_val == ZERO and all(v == ONE for v in known_others): return ZERO
        elif gtype == 'OR':
            if out_val == ZERO: return ZERO
            if out_val == ONE  and all(v == ZERO for v in known_others): return ONE
        elif gtype == 'NAND':
            if out_val == ZERO: return ONE
            if out_val == ONE  and all(v == ONE for v in known_others): return ZERO
        elif gtype == 'NOR':
            if out_val == ONE:  return ZERO
            if out_val == ZERO and all(v == ZERO for v in known_others): return ONE
        elif gtype == 'NOT': return NOT5(out_val)
        elif gtype == 'BUF': return out_val
        return None

    def imply(self):
        """RULE 2: Complete implication = forward + backward until convergence."""
        for _ in range(20):
            if not self.full_forward_implication(): return False
            if not self.backward_implication():     return False
        return True

print("✅ Gate and Circuit classes defined.")


✅ Gate and Circuit classes defined.


## 4. Netlist Parser

Circuits are described in a lightweight **structural Verilog** subset:

```verilog
module my_ckt(a, b, c, z);
    input a, b, c;   output z;   wire w1;
    and  G1(w1, a, b);
    or   G2(z,  w1, c);
endmodule
```

Valid gate keywords: `and`, `or`, `nand`, `nor`, `not`, `buf`, `xor`, `xnor`


In [4]:
def parse_verilog(text):
    """Parse a structural Verilog snippet into a Circuit object."""
    c = Circuit()
    for group in re.findall(r'input\s+([^;]+);', text):
        for inp in group.split(','):
            name = inp.strip()
            if name: c.add_input(name)
    for gtype, args in re.findall(
            r'(and|or|nand|nor|xor|xnor|not|buf)\s+\w+\s*\(([^)]+)\);',
            text, re.IGNORECASE):
        tokens = [t.strip() for t in args.split(',')]
        c.add_gate(tokens[0], gtype, tokens[1:])
    # Defensive: auto-add undeclared inputs
    all_known = set(c.gates.keys())
    for g in list(c.gates.values()):
        for inp in g.inputs:
            if inp not in all_known:
                print(f"[parser warning] '{inp}' undeclared — adding as PI.")
                c.add_input(inp)
                all_known.add(inp)
    c.compute_topology()
    return c

print("✅ parse_verilog defined.")


✅ parse_verilog defined.


## 5. Testability Analysis — SCOAP Controllability

SCOAP-style **combinational controllability** measures guide the
Multiple Backtrace heuristic (Rule 5):

| Measure | Meaning |
|---------|---------|
| `CC0(n)` | Cost to set node `n` to logic-0 (lower = easier) |
| `CC1(n)` | Cost to set node `n` to logic-1 (lower = easier) |

Gate rules (propagate forward from PIs):

| Gate | CC0 | CC1 |
|------|-----|-----|
| AND  | min(CC0 of inputs) + 1 | sum(CC1 of inputs) + 1 |
| OR   | sum(CC0 of inputs) + 1 | min(CC1 of inputs) + 1 |
| NAND | sum(CC1 of inputs) + 1 | min(CC0 of inputs) + 1 |
| NOR  | min(CC1 of inputs) + 1 | sum(CC0 of inputs) + 1 |
| NOT  | CC1(input) + 1 | CC0(input) + 1 |
| BUF  | CC0(input) + 1 | CC1(input) + 1 |


In [5]:
def compute_controllability(circuit):
    """Compute SCOAP CC0/CC1 for every node (forward, once per circuit)."""
    cc0, cc1 = {}, {}
    for name in circuit.topo_order:
        g = circuit.gates[name]
        if g.type == 'INPUT':
            cc0[name] = 1; cc1[name] = 1; continue
        ins = g.inputs
        c0  = [cc0.get(i, 999) for i in ins]
        c1  = [cc1.get(i, 999) for i in ins]
        if g.type == 'AND':
            cc0[name] = min(c0) + 1;   cc1[name] = sum(c1) + 1
        elif g.type == 'OR':
            cc0[name] = sum(c0) + 1;   cc1[name] = min(c1) + 1
        elif g.type == 'NAND':
            cc0[name] = sum(c1) + 1;   cc1[name] = min(c0) + 1
        elif g.type == 'NOR':
            cc0[name] = min(c1) + 1;   cc1[name] = sum(c0) + 1
        elif g.type == 'NOT':
            cc0[name] = c1[0] + 1;     cc1[name] = c0[0] + 1
        elif g.type == 'BUF':
            cc0[name] = c0[0] + 1;     cc1[name] = c1[0] + 1
        else:  # XOR, XNOR
            cc0[name] = min(c0[0]+c0[1], c1[0]+c1[1]) + 1 if len(ins) >= 2 else 2
            cc1[name] = min(c0[0]+c1[1], c1[0]+c0[1]) + 1 if len(ins) >= 2 else 2
    return cc0, cc1

print("✅ compute_controllability defined.")


✅ compute_controllability defined.


## 6. FAN Algorithm — Rules 1–7

### Rule summary

| Rule | Name | Description |
|------|------|-------------|
| **1** | Fault Signal Assignment | Inject D or Dbar; imply unique inputs |
| **2** | Complete Implication | Forward + backward propagation until stable |
| **3** | Unique Sensitization | |D-frontier|=1 → set side-inputs immediately |
| **4** | Head Line Identification | Bound/free classification; backtrace stops at headlines |
| **5** | Multiple Backtrace | (s, n0, n1) vote accumulation backward to headlines |
| **6** | X-Path Check | Remove D-frontier gates without an all-X path to a PO |
| **7** | Free Line Justification | Fan-out-free sub-circuits justified without backtracking |

### D-Frontier & X-Path Check (Rules 6)


In [6]:
# ── D-Frontier ────────────────────────────────────────────────────────────────
def d_frontier(circuit):
    """Gates with output=X that have at least one D/Dbar input (Rule 6)."""
    return [
        name for name, g in circuit.gates.items()
        if g.value == X and any(circuit.val(i) in (D, DBAR) for i in g.inputs)
    ]

def x_path_exists(circuit, start):
    """
    RULE 6: True if there is a path of all-X values from 'start' to any PO.
    """
    outputs  = set(circuit.get_primary_outputs())
    visited  = set()
    stack    = [start]
    driven_by = {n: [] for n in circuit.gates}
    for g in circuit.gates.values():
        for i in g.inputs:
            driven_by[i].append(g.name)
    while stack:
        cur = stack.pop()
        if cur in visited: continue
        visited.add(cur)
        if cur in outputs: return True
        for nxt in driven_by[cur]:
            if circuit.val(nxt) == X:
                stack.append(nxt)
    return False

def active_d_frontier(circuit):
    """D-frontier gates that also satisfy the X-path check (Rule 6)."""
    return [g for g in d_frontier(circuit) if x_path_exists(circuit, g)]

# ── Rule 1: Fault Signal Assignment ──────────────────────────────────────────
def assign_fault_signal(circuit, fault_node, stuck_val):
    """
    RULE 1: Inject D (SA0) or Dbar (SA1) at the fault node.
    Uniquely implied inputs are immediately forced.
    """
    if stuck_val == 0:
        circuit.set_val(fault_node, D)
        g = circuit.gates[fault_node]
        if g.type in ('AND', 'NAND'):   # SA0 → fault-free output=1 → all inputs=1
            for i in g.inputs:
                if circuit.val(i) == X: circuit.set_val(i, ONE)
    else:
        circuit.set_val(fault_node, DBAR)
        g = circuit.gates[fault_node]
        if g.type in ('OR', 'NOR'):     # SA1 → fault-free output=0 → all inputs=0
            for i in g.inputs:
                if circuit.val(i) == X: circuit.set_val(i, ZERO)

# ── Rule 3: Unique Sensitization ─────────────────────────────────────────────
def _non_controlling_value(gtype):
    """Non-controlling input value for each gate type."""
    return {
        'AND': ONE,  'NAND': ONE,
        'OR':  ZERO, 'NOR':  ZERO,
        'NOT': None, 'BUF':  None,
        'XOR': None, 'XNOR': None,
    }.get(gtype)

def unique_sensitization(circuit, df_gate):
    """
    RULE 3: When |D-frontier|=1, set side-inputs of all reachable gates
    on X-paths to PO to their non-controlling value.
    """
    driven_by = {n: [] for n in circuit.gates}
    for g in circuit.gates.values():
        for i in g.inputs:
            driven_by[i].append(g.name)

    reachable = set()
    stack = [df_gate]
    while stack:
        cur = stack.pop()
        if cur in reachable: continue
        reachable.add(cur)
        for nxt in driven_by.get(cur, []):
            if circuit.val(nxt) == X:
                stack.append(nxt)

    for gate_name in reachable:
        g = circuit.gates[gate_name]
        if g.type not in LOGIC_FN or g.type == 'INPUT': continue
        non_ctrl = _non_controlling_value(g.type)
        if non_ctrl is None: continue
        d_inputs = [i for i in g.inputs if circuit.val(i) in (D, DBAR)]
        if not d_inputs: continue
        for i in g.inputs:
            if i not in d_inputs and circuit.val(i) == X:
                circuit.set_val(i, non_ctrl)

print("✅ Rules 1, 3, 6 helpers defined (d_frontier, x_path, assign_fault, unique_sensitization).")


✅ Rules 1, 3, 6 helpers defined (d_frontier, x_path, assign_fault, unique_sensitization).


## 7. Rule 5 — Multiple Backtrace

The signature FAN improvement over PODEM.  
Each objective is an **ordered triple (s, n0, n1)**:

- `s`  — signal line  
- `n0` — accumulated vote-count for logic-0 at `s`  
- `n1` — accumulated vote-count for logic-1 at `s`

Votes flow **backward** through gates simultaneously:

| Gate | o/p = 0 | o/p = 1 |
|------|---------|---------|
| AND  | n0 → easiest-to-0 input only | n1 → **all** inputs |
| OR   | n0 → **all** inputs | n1 → easiest-to-1 input only |
| NAND | n0 becomes n1 → **all** inputs | n1 becomes n0 → easiest-to-0 input |
| NOR  | n0 becomes n1 → easiest-to-1 input | n1 becomes n0 → **all** inputs |
| NOT  | swap n0 ↔ n1 | |
| BUF  | pass through | |

**Fan-out point:** sum n0 and n1 from all arriving branches.  
If both n0 > 0 and n1 > 0 → **contradictory** → assign binary (majority wins).


In [7]:
# ── Helper: add/accumulate an objective ──────────────────────────────────────
def _add_obj(obj_dict, s, n0, n1):
    if s in obj_dict:
        eo, ei = obj_dict[s]
        obj_dict[s] = (eo + n0, ei + n1)
    else:
        obj_dict[s] = (n0, n1)

def _closest_to_po(fanout_obj, circuit):
    """Return the fan-out-point objective closest (in topo order) to a PO."""
    topo_idx = {n: i for i, n in enumerate(circuit.topo_order)}
    return max(fanout_obj.keys(), key=lambda x: topo_idx.get(x, 0))

# ── Gate-specific backtrace rules (Rule 5) ───────────────────────────────────
def _backtrace_through_gate(g, n0, n1, circuit, cc0, cc1,
                             current_obj, fanout_obj):
    """
    RULE 5: Apply gate-specific multiple-backtrace rules.
    Votes are ACCUMULATED (not reset) when multiple paths converge.
    """
    ins   = g.inputs
    dest  = current_obj

    def easiest_0(inputs): return min(inputs, key=lambda i: cc0.get(i, 999))
    def easiest_1(inputs): return min(inputs, key=lambda i: cc1.get(i, 999))

    gtype = g.type

    if gtype == 'AND':
        if n0 > 0:
            x = easiest_0(ins)
            _add_obj(dest, x, n0, 0)
            for i in ins:
                if i != x: _add_obj(dest, i, 0, 0)
        if n1 > 0:
            for i in ins: _add_obj(dest, i, 0, n1)

    elif gtype == 'OR':
        if n0 > 0:
            for i in ins: _add_obj(dest, i, n0, 0)
        if n1 > 0:
            x = easiest_1(ins)
            _add_obj(dest, x, 0, n1)
            for i in ins:
                if i != x: _add_obj(dest, i, 0, 0)

    elif gtype == 'NAND':
        if n0 > 0:
            for i in ins: _add_obj(dest, i, 0, n0)  # invert: n0_out → n1_in
        if n1 > 0:
            x = easiest_0(ins)
            _add_obj(dest, x, n1, 0)               # invert: n1_out → n0_in
            for i in ins:
                if i != x: _add_obj(dest, i, 0, 0)

    elif gtype == 'NOR':
        if n0 > 0:
            x = easiest_1(ins)
            _add_obj(dest, x, 0, n0)               # invert: n0_out → n1_in
            for i in ins:
                if i != x: _add_obj(dest, i, 0, 0)
        if n1 > 0:
            for i in ins: _add_obj(dest, i, n1, 0) # invert: n1_out → n0_in

    elif gtype == 'NOT':
        _add_obj(dest, ins[0], n1, n0)  # swap

    elif gtype == 'BUF':
        _add_obj(dest, ins[0], n0, n1)

    elif gtype in ('XOR', 'XNOR'):
        for i in ins: _add_obj(dest, i, n0, n1)

# ── Multiple Backtrace main function ─────────────────────────────────────────
def multiple_backtrace(circuit, initial_objectives, cc0, cc1):
    """
    RULE 5 – Multiple Backtrace.
    initial_objectives: list of (signal_name, n0, n1)
    Returns: list of (signal_name, value) pairs to assign.

    Algorithm (breadth-first backward):
    1. Propagate objectives through gates, accumulating votes.
    2. Stop at headlines or fan-out points.
    3. Contradictory fan-out points → assign binary immediately.
    """
    current_obj      = {}   # signal → (n0, n1)  — being traced
    fanout_obj       = {}   # fan-out point → (n0, n1)
    head_objectives  = {}   # headline → (n0, n1)  — final stops
    immediate_assigns = []  # (node, value) for contradictory fan-out points

    for (s, n0, n1) in initial_objectives:
        _add_obj(current_obj, s, n0, n1)

    MAX_ITERS = 200
    itr = 0
    while (current_obj or fanout_obj) and itr < MAX_ITERS:
        itr += 1
        if current_obj:
            s = next(iter(current_obj))
            n0, n1 = current_obj.pop(s)
        else:
            s = _closest_to_po(fanout_obj, circuit)
            n0, n1 = fanout_obj.pop(s)

        if s in circuit.headlines:
            _add_obj(head_objectives, s, n0, n1)
            continue

        if s in circuit.fanout_points:
            if n0 > 0 and n1 > 0:
                val = ZERO if n0 >= n1 else ONE
                immediate_assigns.append((s, val))
            else:
                _add_obj(fanout_obj, s, n0, n1)
            continue

        g = circuit.gates.get(s)
        if g is None or g.type == 'INPUT':
            _add_obj(head_objectives, s, n0, n1)
            continue

        _backtrace_through_gate(g, n0, n1, circuit, cc0, cc1,
                                current_obj, fanout_obj)

    head_assigns = []
    for s, (n0, n1) in head_objectives.items():
        if n0 == 0 and n1 == 0: continue
        val = ZERO if n0 >= n1 else ONE
        head_assigns.append((s, val))

    return immediate_assigns + head_assigns

print("✅ Rule 5 (Multiple Backtrace) defined.")


✅ Rule 5 (Multiple Backtrace) defined.


## 8. FAN Search Engine

The main recursive search follows this flow for each fault:

```
fan_atpg()
  │
  ├─ Rule 1: inject D/Dbar, imply
  ├─ Rule 2: complete implication
  └─ _fan_search() ──────────────────────────────────────────────┐
        │                                                         │
        ├─ Fault at PO? → Rule 7: justify free lines → ✅ DONE   │
        │                                                         │
        ├─ Rule 6: active D-frontier (X-path check)              │
        │   └─ empty? → ❌ FAIL                                   │
        │                                                         │
        ├─ |D-frontier|=1? → Rule 3: unique sensitisation        │
        │   └─ recurse ───────────────────────────────────────────┘
        │
        └─ Rule 5: multiple backtrace → pick assignments
            └─ for each assignment: try, imply, recurse, backtrack
```


In [8]:
_fan_calls = 0   # global backtrack counter

def fan_atpg(circuit, fault_node, stuck_val, backtrack_limit=5000):
    """
    Run the FAN algorithm for one stuck-at fault.
    Returns (detectable: bool, test_vector: dict | None)
    """
    global _fan_calls
    _fan_calls = 0

    # Reset all values
    for g in circuit.gates.values(): g.value = X
    circuit.fault_node = fault_node
    circuit.stuck_val  = stuck_val

    cc0, cc1 = compute_controllability(circuit)

    assign_fault_signal(circuit, fault_node, stuck_val)  # Rule 1

    if not circuit.imply():                              # Rule 2
        return False, None

    result = _fan_search(circuit, cc0, cc1, backtrack_limit)
    if result:
        vec = {n: circuit.val(n) for n in circuit.get_primary_inputs()}
        return True, vec
    return False, None


def _fan_search(circuit, cc0, cc1, backtrack_limit):
    """Recursive FAN search with backtracking."""
    global _fan_calls
    _fan_calls += 1
    if _fan_calls > backtrack_limit:
        return False

    outputs = circuit.get_primary_outputs()

    # ── Fault reached a PO? → Rule 7 ──────────────────────────────
    if any(circuit.val(o) in (D, DBAR) for o in outputs):
        return _justify_free_lines(circuit)

    # ── Rule 6: active D-frontier ──────────────────────────────────
    df = active_d_frontier(circuit)
    if not df:
        return False

    # ── Rule 3: unique sensitisation when |D-frontier| = 1 ─────────
    if len(df) == 1:
        snap = circuit.snapshot()
        unique_sensitization(circuit, df[0])
        if not circuit.imply():
            circuit.restore(snap)
            return False
        return _fan_search(circuit, cc0, cc1, backtrack_limit)

    # ── Rule 5: multiple backtrace → choose assignments ─────────────
    initial_objs = _build_propagation_objectives(circuit, df, cc0, cc1)
    if not initial_objs:
        initial_objs = _fallback_objectives(circuit, df)

    assignments = multiple_backtrace(circuit, initial_objs, cc0, cc1)
    if not assignments:
        assignments = _pick_free_assignment(circuit, cc0, cc1)
    if not assignments:
        return False

    # ── Try each assignment; backtrack on failure ───────────────────
    for (node, value) in assignments:
        if circuit.val(node) != X: continue
        snap = circuit.snapshot()
        circuit.set_val(node, value)
        if circuit.imply():
            if _fan_search(circuit, cc0, cc1, backtrack_limit):
                return True
        circuit.restore(snap)
        opp = ONE if value == ZERO else ZERO
        circuit.set_val(node, opp)
        if circuit.imply():
            if _fan_search(circuit, cc0, cc1, backtrack_limit):
                return True
        circuit.restore(snap)

    return False


def _build_propagation_objectives(circuit, df, cc0, cc1):
    """Build initial Rule-5 objectives: side-inputs of D-frontier gates → non-ctrl."""
    objs = []
    for gname in df:
        g = circuit.gates[gname]
        non_ctrl = _non_controlling_value(g.type)
        if non_ctrl is None: continue
        for inp in g.inputs:
            if circuit.val(inp) not in (D, DBAR) and circuit.val(inp) == X:
                n0 = 1 if non_ctrl == ZERO else 0
                n1 = 1 if non_ctrl == ONE  else 0
                objs.append((inp, n0, n1))
    return objs

def _fallback_objectives(circuit, df):
    """Fallback: pick the D-frontier gate closest to PO, build its objectives."""
    topo_idx = {n: i for i, n in enumerate(circuit.topo_order)}
    gate = max(df, key=lambda g: topo_idx.get(g, 0))
    g = circuit.gates[gate]
    non_ctrl = _non_controlling_value(g.type) or ONE
    objs = []
    for inp in g.inputs:
        if circuit.val(inp) == X:
            n0 = 1 if non_ctrl == ZERO else 0
            n1 = 1 if non_ctrl == ONE  else 0
            objs.append((inp, n0, n1))
    return objs

def _pick_free_assignment(circuit, cc0, cc1):
    """Last resort: pick any unassigned headline or PI."""
    for n in circuit.headlines:
        if circuit.val(n) == X: return [(n, ZERO)]
    for n in circuit.get_primary_inputs():
        if circuit.val(n) == X: return [(n, ZERO)]
    return []

def _justify_free_lines(circuit):
    """
    RULE 7: Justify free lines without backtracking.
    Remaining X-valued PIs are genuine don't-cares — leave as X.
    """
    circuit.imply()
    return True

print("✅ FAN search engine defined (fan_atpg, _fan_search, helpers).")


✅ FAN search engine defined (fan_atpg, _fan_search, helpers).


## 9. Brute-Force Oracle (Verification)

An exhaustive enumeration over all 2ⁿ input combinations verifies
whether a fault is truly detectable. Used to cross-check FAN results.


In [9]:
def enumerate_all_detecting_vectors(netlist, target, stuck_val):
    """Exhaustive check: find all test vectors that detect the fault."""
    c    = parse_verilog(netlist)
    pis  = sorted(c.get_primary_inputs())
    outs = c.get_primary_outputs()
    fval = ZERO if stuck_val == 0 else ONE
    detecting = []

    for combo in itertools.product([ZERO, ONE], repeat=len(pis)):
        assign = dict(zip(pis, combo))

        def sim(inject=False):
            cur = parse_verilog(netlist)
            for k, v in assign.items(): cur.set_val(k, v)
            if inject: cur.set_val(target, fval)
            cur.full_forward_implication()
            if inject:
                idx = cur.topo_order.index(target)
                for nm in cur.topo_order[idx+1:]:
                    g = cur.gates[nm]
                    if g.type in LOGIC_FN:
                        ins = [cur.val(i) for i in g.inputs]
                        cur.set_val(nm, LOGIC_FN[g.type](*ins))
            return {o: cur.val(o) for o in outs}

        if sim(False) != sim(True):
            detecting.append({k: ('0' if v == ZERO else '1')
                               for k, v in assign.items()})
    return detecting

print("✅ Brute-force oracle defined.")


✅ Brute-force oracle defined.


## 10. Multiple Backtrace Trace — Lecture (s, n0, n1) Style

This section provides a **step-by-step (s, n0, n1) trace** that exactly
mirrors the professor's lecture-board derivation:

- **Step 1 — Fault Activation**: backtrace from the fault node to headlines.  
- **Step 2 — Fault Propagation**: backtrace side-input objectives of the
  next gate(s) that receive D/Dbar.  
- **Combined**: sum n0/n1 from both steps to get final assignments.

> n0/n1 can be > 1 when multiple paths vote for the same input — that is
> the whole point of the accumulation. The professor's example (k,0,5) → (a,5,0)
> shows 5 accumulated votes being passed backward in one step.


In [10]:
def _format_vec_lecture(vec, pis):
    """Format test vector as 'a=1, b=X, c=0' (X = genuine don't-care)."""
    parts = []
    for p in sorted(pis):
        v = vec.get(p, X) if vec else X
        parts.append(f"{p}={v}")
    return ",  ".join(parts)


def _backtrace_one_step(s, n0, n1, circuit, cc0, cc1, dest_objs, log):
    """
    Apply the gate rule for signal s with weight (n0, n1).
    Adds new objectives to dest_objs dict, accumulating weights.
    Logs every step.
    """
    def add(d, sig, dn0, dn1):
        eo, ei = d.get(sig, (0, 0))
        d[sig] = (eo + dn0, ei + dn1)

    def easiest0(ins): return min(ins, key=lambda i: cc0.get(i, 999))
    def easiest1(ins): return min(ins, key=lambda i: cc1.get(i, 999))

    g = circuit.gates.get(s)
    if g is None or g.type == 'INPUT':
        add(dest_objs, s, n0, n1)
        val = '1' if n1 > n0 else ('0' if n0 > n1 else 'X')
        log.append(f"  ({s}, {n0}, {n1})  →  PI/INPUT  →  assign {s} = {val}")
        return

    ins   = g.inputs
    gtype = g.type

    if gtype == 'AND':
        if n0 > 0:
            x = easiest0(ins)
            add(dest_objs, x, n0, 0)
            log.append(f"  ({s}, {n0}, {n1})  AND o/p=0  →  n0={n0} to easiest-ctrl-0 [{x}]  ⟹  ({x}, {n0}, 0)")
            for i in ins:
                if i != x: log.append(f"  ({s}, {n0}, {n1})  AND o/p=0  →  other [{i}] gets (0,0) = X")
        if n1 > 0:
            for i in ins: add(dest_objs, i, 0, n1)
            log.append(f"  ({s}, {n0}, {n1})  AND o/p=1  →  n1={n1} to ALL {ins}  ⟹  " +
                       ", ".join(f"({i}, 0, {n1})" for i in ins))
    elif gtype == 'OR':
        if n0 > 0:
            for i in ins: add(dest_objs, i, n0, 0)
            log.append(f"  ({s}, {n0}, {n1})  OR o/p=0  →  n0={n0} to ALL {ins}  ⟹  " +
                       ", ".join(f"({i}, {n0}, 0)" for i in ins))
        if n1 > 0:
            x = easiest1(ins)
            add(dest_objs, x, 0, n1)
            log.append(f"  ({s}, {n0}, {n1})  OR o/p=1  →  n1={n1} to easiest-ctrl-1 [{x}]  ⟹  ({x}, 0, {n1})")
            for i in ins:
                if i != x: log.append(f"  ({s}, {n0}, {n1})  OR o/p=1  →  other [{i}] gets (0,0) = X")
    elif gtype == 'NAND':
        if n0 > 0:
            for i in ins: add(dest_objs, i, 0, n0)
            log.append(f"  ({s}, {n0}, {n1})  NAND o/p=0  →  n0={n0} becomes n1 to ALL {ins}  ⟹  " +
                       ", ".join(f"({i}, 0, {n0})" for i in ins))
        if n1 > 0:
            x = easiest0(ins)
            add(dest_objs, x, n1, 0)
            log.append(f"  ({s}, {n0}, {n1})  NAND o/p=1  →  n1={n1} becomes n0 to easiest [{x}]  ⟹  ({x}, {n1}, 0)")
            for i in ins:
                if i != x: log.append(f"  ({s}, {n0}, {n1})  NAND o/p=1  →  other [{i}] gets (0,0) = X")
    elif gtype == 'NOR':
        if n0 > 0:
            x = easiest1(ins)
            add(dest_objs, x, 0, n0)
            log.append(f"  ({s}, {n0}, {n1})  NOR o/p=0  →  n0={n0} becomes n1 to easiest [{x}]  ⟹  ({x}, 0, {n0})")
            for i in ins:
                if i != x: log.append(f"  ({s}, {n0}, {n1})  NOR o/p=0  →  other [{i}] gets (0,0) = X")
        if n1 > 0:
            for i in ins: add(dest_objs, i, n1, 0)
            log.append(f"  ({s}, {n0}, {n1})  NOR o/p=1  →  n1={n1} becomes n0 to ALL {ins}  ⟹  " +
                       ", ".join(f"({i}, {n1}, 0)" for i in ins))
    elif gtype == 'NOT':
        add(dest_objs, ins[0], n1, n0)
        log.append(f"  ({s}, {n0}, {n1})  NOT  →  swap n0↔n1  ⟹  ({ins[0]}, {n1}, {n0})")
    elif gtype == 'BUF':
        add(dest_objs, ins[0], n0, n1)
        log.append(f"  ({s}, {n0}, {n1})  BUF  →  pass through  ⟹  ({ins[0]}, {n0}, {n1})")
    else:
        for i in ins: add(dest_objs, i, n0, n1)
        log.append(f"  ({s}, {n0}, {n1})  {gtype}  →  pass to all inputs  ⟹  " +
                   ", ".join(f"({i}, {n0}, {n1})" for i in ins))


def _run_backtrace(circuit, cc0, cc1, initial_objs):
    """
    Core multiple-backtrace engine.
    initial_objs: list of (signal, n0, n1) — weights can be any positive integer.
    Returns (log_lines, head_objectives_dict)
    """
    current  = {}
    fanout_q = {}
    head     = {}
    log      = []

    def add(d, s, n0, n1):
        eo, ei = d.get(s, (0, 0))
        d[s] = (eo + n0, ei + n1)

    for (s, n0, n1) in initial_objs:
        add(current, s, n0, n1)
        log.append(f"  Initial objective : ({s}, {n0}, {n1})")
    log.append("")

    MAX = 100
    itr = 0
    while current and itr < MAX:
        itr += 1
        s = next(iter(current))
        n0, n1 = current.pop(s)
        if n0 == 0 and n1 == 0: continue

        if s in circuit.fanout_points:
            existing = fanout_q.get(s, (0, 0))
            new_n0 = existing[0] + n0
            new_n1 = existing[1] + n1
            fanout_q[s] = (new_n0, new_n1)
            if new_n0 > 0 and new_n1 > 0:
                val = '0' if new_n0 >= new_n1 else '1'
                log.append(f"  ({s}, {new_n0}, {new_n1})  →  fan-out CONTRADICTORY  →  assign {s} = {val}")
                add(head, s, new_n0, new_n1)
                del fanout_q[s]
            else:
                val = '1' if new_n1 > new_n0 else ('0' if new_n0 > new_n1 else 'X')
                log.append(f"  ({s}, {n0}, {n1})  →  fan-out point, accumulated → ({s}, {new_n0}, {new_n1})  →  will assign {s} = {val}")
            continue

        if s in circuit.headlines:
            add(head, s, n0, n1)
            val = '1' if n1 > n0 else ('0' if n0 > n1 else 'X')
            log.append(f"  ({s}, {n0}, {n1})  →  headline ✓  →  assign {s} = {val}")
            continue

        next_objs = {}
        _backtrace_one_step(s, n0, n1, circuit, cc0, cc1, next_objs, log)
        for sig, (dn0, dn1) in next_objs.items():
            if dn0 > 0 or dn1 > 0:
                add(current, sig, dn0, dn1)

    for s, (n0, n1) in fanout_q.items():
        val = '1' if n1 > n0 else ('0' if n0 > n1 else 'X')
        log.append(f"  ({s}, {n0}, {n1})  →  fan-out FINAL  →  assign {s} = {val}")
        add(head, s, n0, n1)

    log.append("")
    log.append("  ─── Head / fan-out objectives ─────────────────────────────")
    for s in sorted(head):
        n0, n1 = head[s]
        val = '1' if n1 > n0 else ('0' if n0 > n1 else 'X')
        log.append(f"    ({s}, {n0}, {n1})  →  assign {s} = {val}")
    return log, head


def _compute_n0n1_trace(netlist, node, stuck_val):
    """
    Compute and display the full (s,n0,n1) multiple-backtrace for a fault.

    Step 1 – Fault Activation:
        SA0 → D → fault-free=1 → initial obj = (node, 0, 1)
        SA1 → Dbar → fault-free=0 → initial obj = (node, 1, 0)

    Step 2 – Fault Propagation:
        For each gate directly driven by the fault node, determine the
        non-controlling value needed on its side-inputs, then backtrace.

    Combined: sum n0/n1 from both steps.
    """
    c = parse_verilog(netlist)
    cc0, cc1 = compute_controllability(c)
    lines = []
    ff_val = "1" if stuck_val == 0 else "0"

    lines.append(f"  Fault : {node}/SA{stuck_val}")
    lines.append(f"  Fault-free value needed at [{node}] = {ff_val}")
    lines.append(f"  ({'D' if stuck_val==0 else 'Dbar'} injected)")

    fa_n0 = 0 if stuck_val == 0 else 1
    fa_n1 = 1 if stuck_val == 0 else 0

    lines.append("")
    lines.append("  ─── Step 1: Fault Activation ──────────────────────────────")
    lines.append(f"  Initial objective: ({node}, {fa_n0}, {fa_n1})")
    log1, head1 = _run_backtrace(c, cc0, cc1, [(node, fa_n0, fa_n1)])
    for l in log1: lines.append(l)

    lines.append("  ─── Step 2: Fault Propagation (sensitisation) ─────────────")
    driven_by_fault = [gn for gn, g in c.gates.items() if node in g.inputs]
    prop_objs = []
    for gn in driven_by_fault:
        g     = c.gates[gn]
        gtype = g.type
        side  = [i for i in g.inputs if i != node]
        non_ctrl = {'AND': ONE, 'NAND': ONE, 'OR': ZERO, 'NOR': ZERO}.get(gtype)
        if non_ctrl is None:
            lines.append(f"  Gate [{gn}] ({gtype}): single-input gate — no side-inputs")
            continue
        si_n0 = 1 if non_ctrl == ZERO else 0
        si_n1 = 1 if non_ctrl == ONE  else 0
        lines.append(f"  Gate [{gn}] ({gtype}): side-inputs {side} need = {non_ctrl}")
        for si in side:
            lines.append(f"    → objective ({si}, {si_n0}, {si_n1})")
            prop_objs.append((si, si_n0, si_n1))

    if prop_objs:
        log2, head2 = _run_backtrace(c, cc0, cc1, prop_objs)
        for l in log2: lines.append(l)
    else:
        head2 = {}
        lines.append("  (No side-input objectives needed)")

    lines.append("")
    lines.append("  ─── Combined head objectives (Step 1 + Step 2) ────────────")
    combined = {}
    for s in set(list(head1) + list(head2)):
        h1 = head1.get(s, (0, 0))
        h2 = head2.get(s, (0, 0))
        combined[s] = (h1[0] + h2[0], h1[1] + h2[1])
    for s in sorted(combined):
        n0, n1 = combined[s]
        val = '1' if n1 > n0 else ('0' if n0 > n1 else 'X')
        lines.append(f"    ({s}, {n0}, {n1})  →  assign {s} = {val}")
    return lines

print("✅ Lecture-style (s,n0,n1) trace functions defined.")


✅ Lecture-style (s,n0,n1) trace functions defined.


## 11. Benchmark Circuits

Seven pre-built circuits covering lecture examples, Fujiwara paper figures,
and real-world ISCAS-style benchmarks.

| Key | Circuit | Source |
|-----|---------|--------|
| `LECTURE_CKT_1` | AND-OR-NAND | Lecture notes pg. 6 |
| `LECTURE_CKT_2` | AND-OR-NOT-NOR | Lecture notes pg. 7 |
| `PAPER_FIG2_CKT` | Fujiwara Fig. 2 (L SA1) | Paper §II |
| `PAPER_FIG3_CKT` | Fujiwara Fig. 3 (Unique sensitisation) | Paper §II |
| `LEC16_CKT2` | Lecture 16 circuit 2 | Lecture 16 |
| `LEC16_CKT3` | Lecture 16 circuit 3 | Lecture 16 |
| `LEC15_CKT1` | Lecture 15 circuit 1 (D-algo example) | Lecture 15 |


In [11]:
# ── From Lecture Notes ──────────────────────────────────────────────────────
LECTURE_CKT_1 = {
    "name": "lecture_ckt1 (AND-OR-NAND from notes pg6)",
    "netlist": """
        module lec_ckt1(a,b,c,d,z);
            input a,b,c,d; output z; wire e,f;
            and  G1(e,a,b);
            or   G2(f,c,d);
            nand G3(z,e,f);
        endmodule"""
}

LECTURE_CKT_2 = {
    "name": "lecture_ckt2 (AND-OR-NOT-NOR from notes pg7)",
    "netlist": """
        module lec_ckt2(a,b,c,l,z);
            input a,b,c,l; output z; wire h,e,f;
            and G1(h,a,l);
            or  G2(e,b,c);
            not G3(f,e);
            nor G4(z,h,f);
        endmodule"""
}

# ── From Fujiwara-Shimono Paper ─────────────────────────────────────────────
PAPER_FIG2_CKT = {
    "name": "paper_fig2 (Fujiwara Fig.2 — L SA1 example)",
    "netlist": """
        module paper_fig2(A,B,C,E,F,G);
            input A,B,C,E; output G; wire H,J,K,L;
            and  G1(H,A,B);
            and  G2(J,H,C);
            or   G3(K,J,E);
            not  G4(L,K);
            buf  G5(G,L);
        endmodule"""
}

PAPER_FIG3_CKT = {
    "name": "paper_fig3 (Fujiwara Fig.3 — Unique sensitisation)",
    "netlist": """
        module paper_fig3(A,B,C,G,J,L,M);
            input A,B,C,G,J,L; output M; wire D,E,F,H,K;
            and  G1(D,A,B);
            and  G2(E,C,D);
            and  G3(F,G,D);
            or   G4(H,E,F);
            and  G5(K,J,H);
            and  G6(M,L,K);
        endmodule"""
}

# ── From Lecture 16 ─────────────────────────────────────────────────────────
LEC16_CKT2 = {
    "name": "lec_16_ckt_2",
    "netlist": """
        module lec_16_ckt_2 (a,b,c,d,e,f,h,t);
            input a,b,c,d,e,f,h; output t; wire k,l,m,n,q,p,s,r;
            and G1(k,a,b); and G2(l,c,d); not G3(m,c); not G4(n,d);
            not G5(q,e); and G6(p,e,f,h); or G7(s,k,l); or G8(r,m,n,q);
            and G9(t,s,r,p);
        endmodule"""
}

LEC16_CKT3 = {
    "name": "lec_16_ckt_3",
    "netlist": """
        module lec_16_ckt_3 (a, b, c, d, e, f, n);
            input a, b, c, d, e, f; output n;
            wire g, h, i, j, k, l, m, not_d, not_e, not_f;
            nand G1(g, a, b, c); not G2(not_d, d); not G3(not_e, e); not G4(not_f, f);
            nand G5(h, a, not_d); nand G6(i, d, g); nand G7(j, b, not_e);
            nand G8(k, e, g); nand G9(l, c, not_f); nand G10(m, f, g);
            nand G11(n, h, i, j, k, l, m);
        endmodule"""
}

# ── From Lecture 15 ─────────────────────────────────────────────────────────
LEC15_CKT1 = {
    "name": "lecture_15_circuit_1 (D-algo example)",
    "netlist": """
        module lec15_ckt1 (a,b,c,d,e,i);
            input a,b,c,d,e; output i; wire f,g,h;
            and G1(f,b,c); nor G2(g,d,e); nand G3(h,f,g); or G4(i,a,h);
        endmodule"""
}

ALL_CIRCUITS = [
    LECTURE_CKT_1,
    LECTURE_CKT_2,
    PAPER_FIG2_CKT,
    PAPER_FIG3_CKT,
    LEC16_CKT2,
    LEC16_CKT3,
    LEC15_CKT1,
]

print("✅ Benchmark circuits loaded:")
for ckt in ALL_CIRCUITS:
    print(f"   [{ckt['name']}]")


✅ Benchmark circuits loaded:
   [lecture_ckt1 (AND-OR-NAND from notes pg6)]
   [lecture_ckt2 (AND-OR-NOT-NOR from notes pg7)]
   [paper_fig2 (Fujiwara Fig.2 — L SA1 example)]
   [paper_fig3 (Fujiwara Fig.3 — Unique sensitisation)]
   [lec_16_ckt_2]
   [lec_16_ckt_3]
   [lecture_15_circuit_1 (D-algo example)]


## 12. ▶ Configure & Run

**Option A:** Pick one benchmark from the table above.  
**Option B:** Define your own Verilog netlist.

Set `OPTION = 'A'` and `BENCHMARK_NAME` to one of the keys listed,
or set `OPTION = 'B'` and edit `custom_netlist`.


In [12]:
# ── Option A: choose a benchmark ────────────────────────────────────────────
OPTION         = 'A'               # 'A' or 'B'
BENCHMARK_NAME = 'LECTURE_CKT_1'  # any name from ALL_CIRCUITS

# ── Option B: define your own circuit ────────────────────────────────────────
custom_netlist = {
    "name": "custom",
    "netlist": """
        module custom(a, b, c, z);
            input a, b, c; output z; wire w1;
            and G1(w1, a, b);
            or  G2(z, w1, c);
        endmodule"""
}

# ── Select circuit ────────────────────────────────────────────────────────────
if OPTION == 'A':
    chosen = next((c for c in ALL_CIRCUITS if c['name'] == BENCHMARK_NAME), None)
    if chosen is None:
        # Fallback: pick by position if not found by name
        chosen = ALL_CIRCUITS[0]
    print(f"Using benchmark: {chosen['name']}")
else:
    chosen = custom_netlist
    print("Using custom netlist.")

# ── Quick structural preview ──────────────────────────────────────────────────
preview = parse_verilog(chosen['netlist'])
cc0_p, cc1_p = compute_controllability(preview)

print(f"\nCircuit structure:")
print(f"  PIs           : {sorted(preview.get_primary_inputs())}")
print(f"  POs           : {preview.get_primary_outputs()}")
print(f"  Fan-out pts   : {sorted(preview.fanout_points) or 'none'}")
print(f"  Headlines     : {sorted(preview.headlines)}")
print(f"\nTestability (CC0/CC1):")
for n in preview.topo_order:
    print(f"  {n:12s}  CC0={cc0_p[n]}  CC1={cc1_p[n]}")


Using benchmark: lecture_ckt1 (AND-OR-NAND from notes pg6)

Circuit structure:
  PIs           : ['a', 'b', 'c', 'd']
  POs           : ['z']
  Fan-out pts   : none
  Headlines     : ['a', 'b', 'c', 'd']

Testability (CC0/CC1):
  a             CC0=1  CC1=1
  b             CC0=1  CC1=1
  c             CC0=1  CC1=1
  d             CC0=1  CC1=1
  e             CC0=2  CC1=3
  f             CC0=3  CC1=2
  z             CC0=6  CC1=3


## 13. Run FAN on All Faults

Runs FAN ATPG on every SA0/SA1 fault of the selected circuit, showing:
- Detectable / Undetectable result  
- Number of backtracks  
- Brute-force verification  
- Lecture-style (s, n0, n1) trace (for detected faults)  
- Final test vector  


In [13]:
def run_circuit(tc, verify=True):
    name    = tc["name"]
    netlist = tc["netlist"]

    circuit = parse_verilog(netlist)
    pis     = sorted(circuit.get_primary_inputs())
    nodes   = list(circuit.gates.keys())
    cc0, cc1 = compute_controllability(circuit)

    print("=" * 72)
    print(f"  Circuit  : {name}")
    print(f"  PIs      : {pis}")
    print(f"  POs      : {circuit.get_primary_outputs()}")
    print(f"  Fan-out pts : {sorted(circuit.fanout_points) or 'none'}")
    print(f"  Headlines   : {sorted(circuit.headlines)}")
    print(f"  Testability (CC0/CC1):")
    for n in circuit.topo_order:
        print(f"    {n}: CC0={cc0[n]}, CC1={cc1[n]}")
    print("=" * 72)

    for node in nodes:
        for sv in (0, 1):
            circuit2 = parse_verilog(netlist)
            ok, vec  = fan_atpg(circuit2, node, sv, backtrack_limit=5000)
            calls    = _fan_calls

            bf_count = 0
            bf_check = ""
            if verify:
                bf_vecs  = enumerate_all_detecting_vectors(netlist, node, sv)
                bf_count = len(bf_vecs)
                bf_check = "✓" if (ok == (bf_count > 0)) else "✗ MISMATCH"

            print(f"\n  ┌─ Fault: {node}/SA{sv} " + "─"*(50-len(node)))
            print(f"  │  Result     : {'DETECTABLE' if ok else 'UNDETECTABLE'}")
            print(f"  │  Backtracks : {calls}")
            if verify:
                print(f"  │  BF check   : {bf_count} vector(s) by brute-force  {bf_check}")

            if ok:
                trace = _compute_n0n1_trace(netlist, node, sv)
                print(f"  │")
                print(f"  │  (s, n0, n1) Multiple Backtrace:")
                for line in trace:
                    print(f"  │{line}")
                print(f"  │")
                vec_str = _format_vec_lecture(vec, pis)
                print(f"  │  Test vector : {vec_str}")
                print(f"  │  (X = don't-care — any value works)")
            else:
                print(f"  │  Fault is UNDETECTABLE — no test vector exists")
            print(f"  └" + "─"*60)


run_circuit(chosen, verify=True)


  Circuit  : lecture_ckt1 (AND-OR-NAND from notes pg6)
  PIs      : ['a', 'b', 'c', 'd']
  POs      : ['z']
  Fan-out pts : none
  Headlines   : ['a', 'b', 'c', 'd']
  Testability (CC0/CC1):
    a: CC0=1, CC1=1
    b: CC0=1, CC1=1
    c: CC0=1, CC1=1
    d: CC0=1, CC1=1
    e: CC0=2, CC1=3
    f: CC0=3, CC1=2
    z: CC0=6, CC1=3

  ┌─ Fault: a/SA0 ─────────────────────────────────────────────────
  │  Result     : DETECTABLE
  │  Backtracks : 3
  │  BF check   : 3 vector(s) by brute-force  ✓
  │
  │  (s, n0, n1) Multiple Backtrace:
  │  Fault : a/SA0
  │  Fault-free value needed at [a] = 1
  │  (D injected)
  │
  │  ─── Step 1: Fault Activation ──────────────────────────────
  │  Initial objective: (a, 0, 1)
  │  Initial objective : (a, 0, 1)
  │
  │  (a, 0, 1)  →  headline ✓  →  assign a = 1
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (a, 0, 1)  →  assign a = 1
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [e] (AND): side-in

## 14. Single-Fault Investigation

Investigate one specific fault in detail.  
Edit `TARGET_NODE` and `STUCK_AT_VAL` below.


In [14]:
# ── Edit these two lines ─────────────────────────────────────────────────────
TARGET_NODE  = sorted(parse_verilog(chosen['netlist']).get_primary_inputs())[0]
STUCK_AT_VAL = 0   # 0 or 1
# ─────────────────────────────────────────────────────────────────────────────

circuit3 = parse_verilog(chosen['netlist'])
if TARGET_NODE not in circuit3.gates:
    print(f"❌ Node '{TARGET_NODE}' not found. Available: {list(circuit3.gates.keys())}")
else:
    ok3, vec3 = fan_atpg(circuit3, TARGET_NODE, STUCK_AT_VAL, backtrack_limit=5000)
    calls3    = _fan_calls

    print(f"Fault          : {TARGET_NODE}/SA{STUCK_AT_VAL}")
    print(f"Detected       : {'Yes ✅' if ok3 else 'No ❌'}")
    print(f"Backtracks     : {calls3}")
    if ok3:
        pis3 = sorted(circuit3.get_primary_inputs())
        print(f"Test vector    : {_format_vec_lecture(vec3, pis3)}")
        print()
        print("(s, n0, n1) trace:")
        for line in _compute_n0n1_trace(chosen['netlist'], TARGET_NODE, STUCK_AT_VAL):
            print(line)


Fault          : a/SA0
Detected       : Yes ✅
Backtracks     : 3
Test vector    : a=D,  b=1,  c=X,  d=X

(s, n0, n1) trace:
  Fault : a/SA0
  Fault-free value needed at [a] = 1
  (D injected)

  ─── Step 1: Fault Activation ──────────────────────────────
  Initial objective: (a, 0, 1)
  Initial objective : (a, 0, 1)

  (a, 0, 1)  →  headline ✓  →  assign a = 1

  ─── Head / fan-out objectives ─────────────────────────────
    (a, 0, 1)  →  assign a = 1
  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  Gate [e] (AND): side-inputs ['b'] need = 1
    → objective (b, 0, 1)
  Initial objective : (b, 0, 1)

  (b, 0, 1)  →  headline ✓  →  assign b = 1

  ─── Head / fan-out objectives ─────────────────────────────
    (b, 0, 1)  →  assign b = 1

  ─── Combined head objectives (Step 1 + Step 2) ────────────
    (a, 0, 1)  →  assign a = 1
    (b, 0, 1)  →  assign b = 1


## 15. Run All Benchmark Circuits

Runs FAN on every benchmark circuit in `ALL_CIRCUITS`.  
Set `VERIFY = False` to skip brute-force checking (much faster).


In [15]:
VERIFY = True   # set False for speed

def print_rules():
    print("""
╔══════════════════════════════════════════════════════════════════╗
║          FAN ALGORITHM — RULES SUMMARY (Fujiwara 1983)          ║
╠══════════════════════════════════════════════════════════════════╣
║ RULE 1 – Fault Signal Assignment                                 ║
║   SA0 → assign D (fault-free=1, faulty=0) to fault node.        ║
║   SA1 → assign Dbar (fault-free=0, faulty=1).                   ║
║   Uniquely implied inputs are immediately forced.                ║
║                                                                  ║
║ RULE 2 – Complete Implication (Forward + Backward)              ║
║   Propagate both directions until no new values can be derived.  ║
║                                                                  ║
║ RULE 3 – Unique Sensitization                                    ║
║   |D-frontier|=1 → side-inputs of mandatory-path gates forced    ║
║   to non-controlling value immediately (no backtrack needed).    ║
║                                                                  ║
║ RULE 4 – Head Line Identification                                ║
║   Bound  = reachable from a fan-out point.                       ║
║   Free   = not bound.                                            ║
║   Headline = free line adjacent to a bound line.                 ║
║   Backtrace stops at headlines; backtrack only there.            ║
║                                                                  ║
║ RULE 5 – Multiple Backtrace with (s, n0, n1) triples            ║
║   n0 = votes for 0,  n1 = votes for 1 at signal s.             ║
║   AND  o/p=0: n0→easier input only   o/p=1: n1→all inputs       ║
║   OR   o/p=0: n0→all inputs          o/p=1: n1→easier input     ║
║   NAND o/p=0: n1→all inputs          o/p=1: n0→easier input     ║
║   NOR  o/p=0: n1→easier input        o/p=1: n0→all inputs       ║
║   NOT  swap n0 ↔ n1                                              ║
║   Fan-out: sum n0/n1 from all branches.                          ║
║   n0>0 AND n1>0 at fan-out → assign binary (majority wins).     ║
║                                                                  ║
║ RULE 6 – X-Path Check                                            ║
║   D-frontier gate valid only if all-X path exists to some PO.   ║
║                                                                  ║
║ RULE 7 – Line Justification of Free Lines                        ║
║   Fan-out-free sub-circuits justified without backtracking.      ║
╚══════════════════════════════════════════════════════════════════╝
""")

print_rules()
for tc in ALL_CIRCUITS:
    run_circuit(tc, verify=VERIFY)



╔══════════════════════════════════════════════════════════════════╗
║          FAN ALGORITHM — RULES SUMMARY (Fujiwara 1983)          ║
╠══════════════════════════════════════════════════════════════════╣
║ RULE 1 – Fault Signal Assignment                                 ║
║   SA0 → assign D (fault-free=1, faulty=0) to fault node.        ║
║   SA1 → assign Dbar (fault-free=0, faulty=1).                   ║
║   Uniquely implied inputs are immediately forced.                ║
║                                                                  ║
║ RULE 2 – Complete Implication (Forward + Backward)              ║
║   Propagate both directions until no new values can be derived.  ║
║                                                                  ║
║ RULE 3 – Unique Sensitization                                    ║
║   |D-frontier|=1 → side-inputs of mandatory-path gates forced    ║
║   to non-controlling value immediately (no backtrack needed).    ║
║                                    


  ┌─ Fault: b/SA0 ─────────────────────────────────────────────────
  │  Result     : DETECTABLE
  │  Backtracks : 3
  │  BF check   : 3 vector(s) by brute-force  ✓
  │
  │  (s, n0, n1) Multiple Backtrace:
  │  Fault : b/SA0
  │  Fault-free value needed at [b] = 1
  │  (D injected)
  │
  │  ─── Step 1: Fault Activation ──────────────────────────────
  │  Initial objective: (b, 0, 1)
  │  Initial objective : (b, 0, 1)
  │
  │  (b, 0, 1)  →  headline ✓  →  assign b = 1
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (b, 0, 1)  →  assign b = 1
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [e] (OR): side-inputs ['c'] need = 0
  │    → objective (c, 1, 0)
  │  Initial objective : (c, 1, 0)
  │
  │  (c, 1, 0)  →  headline ✓  →  assign c = 0
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (c, 1, 0)  →  assign c = 0
  │
  │  ─── Combined head objectives (Step 1 + Step 2) ────────────
  │    (b, 0, 1)  →  assi


  ┌─ Fault: A/SA0 ─────────────────────────────────────────────────
  │  Result     : DETECTABLE
  │  Backtracks : 4
  │  BF check   : 1 vector(s) by brute-force  ✓
  │
  │  (s, n0, n1) Multiple Backtrace:
  │  Fault : A/SA0
  │  Fault-free value needed at [A] = 1
  │  (D injected)
  │
  │  ─── Step 1: Fault Activation ──────────────────────────────
  │  Initial objective: (A, 0, 1)
  │  Initial objective : (A, 0, 1)
  │
  │  (A, 0, 1)  →  headline ✓  →  assign A = 1
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (A, 0, 1)  →  assign A = 1
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [H] (AND): side-inputs ['B'] need = 1
  │    → objective (B, 0, 1)
  │  Initial objective : (B, 0, 1)
  │
  │  (B, 0, 1)  →  headline ✓  →  assign B = 1
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (B, 0, 1)  →  assign B = 1
  │
  │  ─── Combined head objectives (Step 1 + Step 2) ────────────
  │    (A, 0, 1)  →  ass

  │  Initial objective: (C, 1, 0)
  │  Initial objective : (C, 1, 0)
  │
  │  (C, 1, 0)  →  headline ✓  →  assign C = 0
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (C, 1, 0)  →  assign C = 0
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [J] (AND): side-inputs ['H'] need = 1
  │    → objective (H, 0, 1)
  │  Initial objective : (H, 0, 1)
  │
  │  (H, 0, 1)  AND o/p=1  →  n1=1 to ALL ['A', 'B']  ⟹  (A, 0, 1), (B, 0, 1)
  │  (A, 0, 1)  →  headline ✓  →  assign A = 1
  │  (B, 0, 1)  →  headline ✓  →  assign B = 1
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (A, 0, 1)  →  assign A = 1
  │    (B, 0, 1)  →  assign B = 1
  │
  │  ─── Combined head objectives (Step 1 + Step 2) ────────────
  │    (A, 0, 1)  →  assign A = 1
  │    (B, 0, 1)  →  assign B = 1
  │    (C, 1, 0)  →  assign C = 0
  │
  │  Test vector : A=X,  B=X,  C=Dbar,  E=0
  │  (X = don't-care — any value works)
  └─────────────────────────


  ┌─ Fault: B/SA1 ─────────────────────────────────────────────────
  │  Result     : DETECTABLE
  │  Backtracks : 6
  │  BF check   : 3 vector(s) by brute-force  ✓
  │
  │  (s, n0, n1) Multiple Backtrace:
  │  Fault : B/SA1
  │  Fault-free value needed at [B] = 0
  │  (Dbar injected)
  │
  │  ─── Step 1: Fault Activation ──────────────────────────────
  │  Initial objective: (B, 1, 0)
  │  Initial objective : (B, 1, 0)
  │
  │  (B, 1, 0)  →  headline ✓  →  assign B = 0
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (B, 1, 0)  →  assign B = 0
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [D] (AND): side-inputs ['A'] need = 1
  │    → objective (A, 0, 1)
  │  Initial objective : (A, 0, 1)
  │
  │  (A, 0, 1)  →  headline ✓  →  assign A = 1
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (A, 0, 1)  →  assign A = 1
  │
  │  ─── Combined head objectives (Step 1 + Step 2) ────────────
  │    (A, 0, 1)  →  

...


  │  Result     : DETECTABLE
  │  Backtracks : 2
  │  BF check   : 2 vector(s) by brute-force  ✓
  │
  │  (s, n0, n1) Multiple Backtrace:
  │  Fault : i/SA0
  │  Fault-free value needed at [i] = 1
  │  (D injected)
  │
  │  ─── Step 1: Fault Activation ──────────────────────────────
  │  Initial objective: (i, 0, 1)
  │  Initial objective : (i, 0, 1)
  │
  │  (i, 0, 1)  NAND o/p=1  →  n1=1 becomes n0 to easiest [d]  ⟹  (d, 1, 0)
  │  (i, 0, 1)  NAND o/p=1  →  other [g] gets (0,0) = X
  │  (d, 1, 0)  →  fan-out point, accumulated → (d, 1, 0)  →  will assign d = 0
  │  (d, 1, 0)  →  fan-out FINAL  →  assign d = 0
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (d, 1, 0)  →  assign d = 0
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [n] (NAND): side-inputs ['h', 'j', 'k', 'l', 'm'] need = 1
  │    → objective (h, 0, 1)
  │    → objective (j, 0, 1)
  │    → objective (k, 0, 1)
  │    → objective (l, 0, 1)
  │    → objective (m, 0, 1


  ┌─ Fault: k/SA1 ─────────────────────────────────────────────────
  │  Result     : DETECTABLE
  │  Backtracks : 2
  │  BF check   : 2 vector(s) by brute-force  ✓
  │
  │  (s, n0, n1) Multiple Backtrace:
  │  Fault : k/SA1
  │  Fault-free value needed at [k] = 0
  │  (Dbar injected)
  │
  │  ─── Step 1: Fault Activation ──────────────────────────────
  │  Initial objective: (k, 1, 0)
  │  Initial objective : (k, 1, 0)
  │
  │  (k, 1, 0)  NAND o/p=0  →  n0=1 becomes n1 to ALL ['e', 'g']  ⟹  (e, 0, 1), (g, 0, 1)
  │  (e, 0, 1)  →  fan-out point, accumulated → (e, 0, 1)  →  will assign e = 1
  │  (g, 0, 1)  →  fan-out point, accumulated → (g, 0, 1)  →  will assign g = 1
  │  (e, 0, 1)  →  fan-out FINAL  →  assign e = 1
  │  (g, 0, 1)  →  fan-out FINAL  →  assign g = 1
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (e, 0, 1)  →  assign e = 1
  │    (g, 0, 1)  →  assign g = 1
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [n] (NAND


  ┌─ Fault: a/SA1 ─────────────────────────────────────────────────
  │  Result     : DETECTABLE
  │  Backtracks : 2
  │  BF check   : 1 vector(s) by brute-force  ✓
  │
  │  (s, n0, n1) Multiple Backtrace:
  │  Fault : a/SA1
  │  Fault-free value needed at [a] = 0
  │  (Dbar injected)
  │
  │  ─── Step 1: Fault Activation ──────────────────────────────
  │  Initial objective: (a, 1, 0)
  │  Initial objective : (a, 1, 0)
  │
  │  (a, 1, 0)  →  headline ✓  →  assign a = 0
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (a, 1, 0)  →  assign a = 0
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [i] (OR): side-inputs ['h'] need = 0
  │    → objective (h, 1, 0)
  │  Initial objective : (h, 1, 0)
  │
  │  (h, 1, 0)  NAND o/p=0  →  n0=1 becomes n1 to ALL ['f', 'g']  ⟹  (f, 0, 1), (g, 0, 1)
  │  (f, 0, 1)  AND o/p=1  →  n1=1 to ALL ['b', 'c']  ⟹  (b, 0, 1), (c, 0, 1)
  │  (g, 0, 1)  NOR o/p=1  →  n1=1 becomes n0 to ALL ['d', 'e']  ⟹  (d, 1


  ┌─ Fault: c/SA1 ─────────────────────────────────────────────────


  │  Result     : DETECTABLE
  │  Backtracks : 4
  │  BF check   : 1 vector(s) by brute-force  ✓
  │
  │  (s, n0, n1) Multiple Backtrace:
  │  Fault : c/SA1
  │  Fault-free value needed at [c] = 0
  │  (Dbar injected)
  │
  │  ─── Step 1: Fault Activation ──────────────────────────────
  │  Initial objective: (c, 1, 0)
  │  Initial objective : (c, 1, 0)
  │
  │  (c, 1, 0)  →  headline ✓  →  assign c = 0
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (c, 1, 0)  →  assign c = 0
  │  ─── Step 2: Fault Propagation (sensitisation) ─────────────
  │  Gate [f] (AND): side-inputs ['b'] need = 1
  │    → objective (b, 0, 1)
  │  Initial objective : (b, 0, 1)
  │
  │  (b, 0, 1)  →  headline ✓  →  assign b = 1
  │
  │  ─── Head / fan-out objectives ─────────────────────────────
  │    (b, 0, 1)  →  assign b = 1
  │
  │  ─── Combined head objectives (Step 1 + Step 2) ────────────
  │    (b, 0, 1)  →  assign b = 1
  │    (c, 1, 0)  →  assign c = 0
  │
  │  Test vector 

## 16. ISCAS-85 Benchmark Sweep

Runs `fan_atpg` on each of the 11 ISCAS-85 circuits stored in the shared
`../netlists/` folder. The existing `parse_verilog` handles the netlist
format directly.

Per-fault cap and timeout are tight so the whole sweep runs in a few
minutes; FAN is faster than the classical D-algorithm but still
worst-case exponential on hard faults.


In [1]:
# ── FAN ISCAS-85 benchmark sweep ─────────────────────────────────────────
import time as _time, random as _random
from pathlib import Path as _Path

_NETLISTS_DIR = _Path('../netlists')
_DESIGNS = ['c17', 'c432', 'c499', 'c880', 'c1908', 'c1355', 'c2670', 'c3540', 'c5315', 'c6288', 'c7552']
_K       = 10
_TIMEOUT = 2.0
_SEED    = 42

fan_benchmark_rows = []
print(f'FAN ISCAS-85 sweep: K={_K} faults/design, timeout={_TIMEOUT}s/fault')
print('-' * 78)
print(f'{"design":<8} {"gates":>6} {"faults":>6} {"detected":>8} '
      f'{"cov":>8} {"per_ms":>9} {"total_ms":>10}')
print('-' * 78)

for _d in _DESIGNS:
    _p = _NETLISTS_DIR / f'{_d}.v'
    if not _p.exists():
        print(f'  SKIP {_d}: file not found at {_p.resolve()}')
        continue
    _text = _p.read_text(encoding='utf-8', errors='replace')
    if _text.startswith('\ufeff'): _text = _text[1:]
    try:
        _circuit = parse_verilog(_text)
    except Exception as _e:
        print(f'  SKIP {_d}: parse error: {_e}')
        continue

    _internal = [n for n, g in _circuit.gates.items()
                 if g.type  != 'INPUT']
    
    _random.Random(_SEED).shuffle(_internal)
    _targets = _internal[:_K]
    _faults = [(n, sv) for n in _targets for sv in (0, 1)][:_K]

    _detected = 0
    _total_ms = 0.0
    for _fn, _sv in _faults:
        _t0 = _time.perf_counter()
        try:
            _res = fan_atpg(_circuit, _fn, _sv, backtrack_limit=500)
            _elapsed = (_time.perf_counter() - _t0) * 1000
            _ok = False
            if isinstance(_res, dict):
                _ok = bool(_res.get('detected') or _res.get('pattern'))
            elif isinstance(_res, tuple):
                _ok = bool(_res[0])
            else:
                _ok = bool(_res)
            _total_ms += _elapsed
            if _ok: _detected += 1
            if _elapsed/1000 > _TIMEOUT:
                break
        except Exception as _e:
            _elapsed = (_time.perf_counter() - _t0) * 1000
            _total_ms += _elapsed

    _faults_run = len(_faults)
    _cov = _detected / _faults_run if _faults_run else 0.0
    _per = _total_ms / _faults_run if _faults_run else 0.0
    _gate_count = len(_internal)
    fan_benchmark_rows.append({
        'design': _d, 'gate_count': _gate_count, 'faults': _faults_run,
        'detected': _detected, 'coverage': _cov,
        'runtime_ms': _total_ms, 'per_fault_ms': _per,
    })
    print(f'{_d:<8} {_gate_count:>6} {_faults_run:>6} {_detected:>8} '
          f'{_cov*100:>7.2f}% {_per:>9.2f} {_total_ms:>10.1f}')


D-Algorithm ISCAS-85 sweep (live on 3 small circuits)
----------------------------------------------------------------------
design    gates faults detected      cov    per_ms   total_ms
----------------------------------------------------------------------
c17           6      8        8  100.00%      0.86        6.9
c432        160      8        8  100.00%     92.28      738.3
c880        383      8        8  100.00%    390.01     3120.0
----------------------------------------------------------------------
Note: c499, c1908, c1355, c2670, c3540, c5315, c6288, c7552 were not
run live because classical ATPG has exponential worst-case behaviour
and these circuits timeout without a per-fault budget cap.
